In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats

SCRIPT_DIR = Path().resolve()
PCS_DIR = SCRIPT_DIR

FEATURE_NAMES = [
    'Vertical_Minor_Seconds','Vertical_Tritones','Vertical_Sevenths',
    'Vertical_Dissonance_Ratio','Standard_Triads','Seventh_Chords',
    'Non-Standard_Chords','Complex_Chords',
    'Distance_Between_Two_Most_Common_Vertical_Intervals',
    'Prevalence_Ratio_of_Two_Most_Common_Vertical_Intervals',
    'Variability_of_Number_of_Simultaneous_Pitch_Classes',
]

LAB = {f:k for f,k in zip(FEATURE_NAMES,
    ['VMS','VT','VS','VDR','ST','7C','NSC','CC','DTMCVI','PRTMCVI','VNSPC'])}
fs = list(LAB.values())

score_map = {
    'A mucho m\u00e1s compleja que B': 2,
    'A m\u00e1s compleja que B': 1,
    'Igual de complejas': 0,
    'B m\u00e1s compleja que A': -1,
    'B mucho m\u00e1s compleja que A': -2
}

In [2]:
df = pd.read_csv(PCS_DIR / 'survey_raw.csv', sep=',')
C_cols = [c for c in df.columns if c.startswith('C') and c != 'Correos']
L_cols = ['L1','L2','L3','L4','L5','L6']
H_cols = ['H1','H2','H3','H4','H5','H6']
all_cols = L_cols + H_cols

for col in all_cols + C_cols:
    df[col + '_score'] = df[col].map(score_map)

mask = (df['C1_score'] < 0) & (df['C2_score'] < 0)
df_f = df[mask].copy()
print(f'Participantes: {len(df)} total, {len(df_f)} pasaron control')

Participantes: 52 total, 43 pasaron control


In [3]:
rvae_files = {
    'L1': ('L1) Cmin-Gmaj-Cmin-Fmin-Cmin.mid',  'M3) Cm-Bb-Ab-Cs-Eb.mid'),
    'L2': ('L4) Cm-C7-Fm-Bb-C.mid',              'M2) Cmin-Gmaj-C7-Fmin-Cmin.mid'),
    'L3': ('L3) Cmin-Gmaj-A7-Fmaj-Cmin.mid',     'M22) Cm-Bb-Bb-Dm7-Eb.mid'),
    'L4': ('L10) Cs-Fsdim7-Fm7-Bb9-F.mid',       'M4) Cmin-Gmaj-Fs7-F7-Cmin.mid'),
    'L5': ('L5) Cmin-Emin-Emin-Fmaj-Cmin.mid',   'M1) Cm-Bb-Bb-Cm7-Fm7.mid'),
    'L6': ('L9) Abmaj7-Cmin7-Csmaj-Ebmaj.mid',   'M6) Cmin-Fsmin-A7-Dsmin-Cmin.mid'),
}
human_files = {
    'H1': ('00001REARM_OG.mid',  '00001REARM_MIO.mid'),
    'H2': ('00002REARM_OG.mid',  '00002REARM_MIO.mid'),
    'H3': ('00003REARM_OG.mid',  '00003REARM_MIO.mid'),
    'H4': ('00004REARM_OG.mid',  '00004REARM_MIO.mid'),
    'H5': ('00005REARM_OG.mid',  '00005REARM_MIO.mid'),
    'H6': ('00006REARM_OG.mid',  '00006REARM_MIO.mid'),
}

def get_labels(col):
    if col in rvae_files:
        return ('RVAE','SCL','L') if int(col[1])%2==1 else ('SCL','RVAE','L')
    else:
        return ('Humano','SCL','H') if int(col[1])%2==1 else ('SCL','Humano','H')

complexity_records = []
for col in all_cols:
    sc = col + '_score'
    lab_a, lab_b, blk = get_labels(col)
    for idx in df_f.index:
        v = df_f.loc[idx, sc]
        if pd.isna(v): continue
        complexity_records.append({'p':idx,'col':col,'blk':blk,'audio':lab_a,'compl':v})
        complexity_records.append({'p':idx,'col':col,'blk':blk,'audio':lab_b,'compl':-v})
df_comp = pd.DataFrame(complexity_records)
print(f'Observaciones de complejidad: {len(df_comp)}')

Observaciones de complejidad: 1032


In [4]:
df_L_all = pd.read_csv(PCS_DIR / 'features_l.csv', index_col=0)
df_L_all.columns = df_L_all.columns.str.strip()
df_L_all.index = df_L_all.index.map(lambda x: Path(str(x).strip()).name)

df_H_all = pd.read_csv(PCS_DIR / 'features_h.csv', index_col=0)
df_H_all.columns = df_H_all.columns.str.strip()
df_H_all.index = df_H_all.index.str.strip()
print(f'Features L: {df_L_all.shape}, H: {df_H_all.shape}')

def get_file(col, audio):
    d = rvae_files if col in rvae_files else human_files
    ref, scl = d[col]
    return scl if audio=='SCL' else ref
def get_df(col):
    return df_L_all if col in rvae_files else df_H_all

comp_by_audio = (df_comp.groupby(['blk','col','audio'])['compl']
    .agg(['mean','std','count','sem']).reset_index()
    .rename(columns={'mean':'compl_mean','std':'compl_std','count':'n','sem':'compl_se'}))

rows = []
for _, r in comp_by_audio.iterrows():
    f = get_file(r['col'], r['audio'])
    dfe = get_df(r['col'])
    if f not in dfe.index: continue
    fr = dfe.loc[f]
    vals = {LAB[fn]: pd.to_numeric(fr[fn], errors='coerce') for fn in FEATURE_NAMES}
    vals['compl'] = r['compl_mean']
    vals['col']=r['col']; vals['audio']=r['audio']; vals['blk']=r['blk']
    rows.append(vals)

df_feats = pd.DataFrame(rows)
print(f'Audios con features: {len(df_feats)}')

Features L: (12, 183), H: (18, 183)
Audios con features: 24


In [5]:
df_f['msi_level'] = pd.qcut(df_f['GOLD_SCORE'], q=3, labels=['Bajo', 'Medio', 'Alto'])

msi_records = []
for col in all_cols:
    lab_a, lab_b, blk = get_labels(col)
    sc = col + '_score'
    for idx in df_f.index:
        v = df_f.loc[idx, sc]
        if pd.isna(v): continue
        msi = df_f.loc[idx, 'msi_level']
        msi_records.append({'msi':msi, 'col':col, 'blk':blk, 'audio':lab_a, 'compl':v})
        msi_records.append({'msi':msi, 'col':col, 'blk':blk, 'audio':lab_b, 'compl':-v})
df_msi = pd.DataFrame(msi_records)

comp_by_msi = df_msi.groupby(['msi','blk','col','audio'])['compl'].mean().reset_index()

def get_file(col, audio):
    d = rvae_files if col in rvae_files else human_files
    ref, scl = d[col]
    return scl if audio=='SCL' else ref
def get_df(col):
    return df_L_all if col in rvae_files else df_H_all
msi_feats_rows = []
for _, r in comp_by_msi.iterrows():
    f = get_file(r['col'], r['audio'])
    dfe = get_df(r['col'])
    if f not in dfe.index: continue
    fr = dfe.loc[f]
    vals = {LAB[fn]: pd.to_numeric(fr[fn], errors='coerce') for fn in FEATURE_NAMES}
    vals['compl'] = r['compl']
    vals['col']=r['col']; vals['audio']=r['audio']; vals['blk']=r['blk']; vals['msi']=r['msi']
    msi_feats_rows.append(vals)
df_msi_feats = pd.DataFrame(msi_feats_rows)

print(f'Total obs por MSI: {df_msi_feats.groupby("msi").size().to_dict()}')

Total obs por MSI: {'Alto': 24, 'Bajo': 24, 'Medio': 24}


In [6]:
msi_corr_rows = []
for msi in ['Bajo', 'Medio', 'Alto']:
    sub = df_msi_feats[df_msi_feats['msi']==msi]
    if len(sub) < 3: continue
    for f in fs:
        r, p = stats.pearsonr(sub[f], sub['compl'])
        msi_corr_rows.append({'MSI':msi, 'Feature':f, 'r':round(r,3), 'p':round(p,4)})

df_msi_corr = pd.DataFrame(msi_corr_rows)

pivot_r = df_msi_corr.pivot(index='Feature', columns='MSI', values='r')
pivot_p = df_msi_corr.pivot(index='Feature', columns='MSI', values='p')

print('Correlaciones calculadas. Exportando...')

Correlaciones calculadas. Exportando...


In [7]:
pivot_r.to_csv(PCS_DIR / 'survey_correlations.csv')
pivot_p.to_csv(PCS_DIR / 'survey_pvalues.csv')
print(f'Exportado: survey_correlations.csv -> {PCS_DIR / "survey_correlations.csv"}')
print(f'Exportado: survey_pvalues.csv -> {PCS_DIR / "survey_pvalues.csv"}')

Exportado: survey_correlations.csv -> /home/pepebeats/SCL_2.0/PCS/survey_correlations.csv
Exportado: survey_pvalues.csv -> /home/pepebeats/SCL_2.0/PCS/survey_pvalues.csv


In [8]:
print(pivot_r.to_string())
print()
print('Significancia:')
pivot_sig = pivot_p.applymap(lambda p: '*' if p < 0.05 else 'n.s.')
print(pivot_sig.to_string())
print()
print('p-values:')
print(pivot_p.to_string())

MSI       Alto   Bajo  Medio
Feature                     
7C       0.580  0.239  0.510
CC      -0.363  0.356 -0.158
DTMCVI   0.325  0.209  0.481
NSC     -0.189  0.240 -0.011
PRTMCVI -0.114  0.079 -0.072
ST      -0.213 -0.382 -0.331
VDR     -0.125  0.406  0.071
VMS      0.107  0.271  0.201
VNSPC    0.475  0.174  0.484
VS      -0.258  0.313 -0.115
VT      -0.119  0.034 -0.109

Significancia:
MSI      Alto  Bajo Medio
Feature                  
7C          *  n.s.     *
CC       n.s.  n.s.  n.s.
DTMCVI   n.s.  n.s.     *
NSC      n.s.  n.s.  n.s.
PRTMCVI  n.s.  n.s.  n.s.
ST       n.s.  n.s.  n.s.
VDR      n.s.     *  n.s.
VMS      n.s.  n.s.  n.s.
VNSPC       *  n.s.     *
VS       n.s.  n.s.  n.s.
VT       n.s.  n.s.  n.s.

p-values:
MSI        Alto    Bajo   Medio
Feature                        
7C       0.0029  0.2611  0.0109
CC       0.0809  0.0877  0.4599
DTMCVI   0.1213  0.3274  0.0173
NSC      0.3753  0.2585  0.9582
PRTMCVI  0.5960  0.7119  0.7382
ST       0.3183  0.0652  0.1144
VD

/tmp/ipykernel_90389/2736156059.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot_sig = pivot_p.applymap(lambda p: '*' if p < 0.05 else 'n.s.')
